In [ ]:
# ================================================================
# ASSIGNMENT 2: LOAN APPROVAL PREDICTION
# LOGISTIC REGRESSION FROM SCRATCH USING NUMPY
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ================================================================
# 1. LOAD DATASET
# ================================================================

DATA_FILE = "loan_approval_dataset.csv"

df = pd.read_csv(DATA_FILE)

# Clean column names because the dataset contains leading spaces
df.columns = df.columns.str.strip()

print("=" * 70)
print("DATASET")
print("=" * 70)

print(f"Shape: {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["loan_status"].value_counts())


# ================================================================
# 2. PREPARE FEATURES AND TARGET
# ================================================================

# loan_id is only an identifier and should not be used as a feature
df = df.drop(columns=["loan_id"])

target = "loan_status"

features = [
    "no_of_dependents",
    "education",
    "self_employed",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value"
]

print("\nFeatures used by the model:")
for feature in features:
    print(" -", feature)


# ================================================================
# 3. HANDLE CATEGORICAL FEATURES
# ================================================================

# Convert categorical variables to numerical dummy variables
X_df = pd.get_dummies(
    df[features],
    columns=["education", "self_employed"],
    drop_first=False,
    dtype=float
)

# Convert target:
# Approved = 1
# Rejected = 0
y = (
    df[target]
    .str.strip()
    .map({
        "Approved": 1,
        "Rejected": 0
    })
    .values
    .astype(float)
)

X = X_df.values.astype(float)

print("\nEncoded model features:")
for feature in X_df.columns:
    print(" -", feature)

print(f"\nTotal numerical model features: {X.shape[1]}")


# ================================================================
# 4. TRAIN / TEST SPLIT FROM SCRATCH
# ================================================================

np.random.seed(42)

indices = np.random.permutation(len(X))

train_size = int(0.80 * len(X))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples : {len(X_test)}")


# ================================================================
# 5. STANDARDIZE FEATURES FROM SCRATCH
# ================================================================

# IMPORTANT:
# Calculate mean and standard deviation ONLY from training data.
# This prevents test-set information from leaking into training.

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Prevent division by zero
std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


# ================================================================
# 6. LOGISTIC REGRESSION FROM SCRATCH
# ================================================================

class LogisticRegressionScratch:

    def __init__(self, learning_rate=0.05, epochs=5000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.loss_history = []

    # ------------------------------------------------------------
    # Sigmoid function
    # ------------------------------------------------------------
    def sigmoid(self, z):

        # Clip values to avoid numerical overflow
        z = np.clip(z, -500, 500)

        return 1 / (1 + np.exp(-z))

    # ------------------------------------------------------------
    # Binary Cross-Entropy Loss
    # ------------------------------------------------------------
    def compute_loss(self, y_true, y_prob):

        epsilon = 1e-15

        y_prob = np.clip(
            y_prob,
            epsilon,
            1 - epsilon
        )

        loss = -np.mean(
            y_true * np.log(y_prob)
            +
            (1 - y_true) * np.log(1 - y_prob)
        )

        return loss

    # ------------------------------------------------------------
    # Training using Gradient Descent
    # ------------------------------------------------------------
    def fit(self, X, y):

        n_samples, n_features = X.shape

        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        print("\n")
        print("=" * 70)
        print("TRAINING LOGISTIC REGRESSION FROM SCRATCH")
        print("=" * 70)

        for epoch in range(self.epochs):

            # ----------------------------------------------------
            # Forward pass
            # ----------------------------------------------------

            # Linear combination
            z = np.dot(X, self.weights) + self.bias

            # Convert linear score to probability
            y_prob = self.sigmoid(z)

            # ----------------------------------------------------
            # Calculate loss
            # ----------------------------------------------------

            loss = self.compute_loss(y, y_prob)

            self.loss_history.append(loss)

            # ----------------------------------------------------
            # Calculate gradients
            # ----------------------------------------------------

            error = y_prob - y

            dw = (1 / n_samples) * np.dot(X.T, error)

            db = (1 / n_samples) * np.sum(error)

            # ----------------------------------------------------
            # Update parameters
            # ----------------------------------------------------

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            # ----------------------------------------------------
            # Display progress
            # ----------------------------------------------------

            if epoch % 500 == 0:
                print(
                    f"Epoch {epoch:5d} | "
                    f"Log Loss: {loss:.6f}"
                )

        print(
            f"Epoch {self.epochs:5d} | "
            f"Log Loss: {self.loss_history[-1]:.6f}"
        )

    # ------------------------------------------------------------
    # Predict probabilities
    # ------------------------------------------------------------
    def predict_proba(self, X):

        z = np.dot(X, self.weights) + self.bias

        return self.sigmoid(z)

    # ------------------------------------------------------------
    # Predict class
    # ------------------------------------------------------------
    def predict(self, X, threshold=0.5):

        probabilities = self.predict_proba(X)

        return (probabilities >= threshold).astype(int)


# ================================================================
# 7. TRAIN MODEL
# ================================================================

model = LogisticRegressionScratch(
    learning_rate=0.05,
    epochs=5000
)

model.fit(X_train, y_train)


# ================================================================
# 8. MAKE PREDICTIONS
# ================================================================

y_prob = model.predict_proba(X_test)

y_pred = model.predict(
    X_test,
    threshold=0.5
)


# ================================================================
# 9. EVALUATION METRICS FROM SCRATCH
# ================================================================

def confusion_matrix_from_scratch(y_true, y_pred):

    true_positive = np.sum(
        (y_true == 1) & (y_pred == 1)
    )

    true_negative = np.sum(
        (y_true == 0) & (y_pred == 0)
    )

    false_positive = np.sum(
        (y_true == 0) & (y_pred == 1)
    )

    false_negative = np.sum(
        (y_true == 1) & (y_pred == 0)
    )

    return (
        true_negative,
        false_positive,
        false_negative,
        true_positive
    )


def accuracy_score_from_scratch(y_true, y_pred):

    return np.mean(y_true == y_pred)


def precision_score_from_scratch(y_true, y_pred):

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    if tp + fp == 0:
        return 0.0

    return tp / (tp + fp)


def recall_score_from_scratch(y_true, y_pred):

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    if tp + fn == 0:
        return 0.0

    return tp / (tp + fn)


def f1_score_from_scratch(y_true, y_pred):

    precision = precision_score_from_scratch(
        y_true,
        y_pred
    )

    recall = recall_score_from_scratch(
        y_true,
        y_pred
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


accuracy = accuracy_score_from_scratch(
    y_test,
    y_pred
)

precision = precision_score_from_scratch(
    y_test,
    y_pred
)

recall = recall_score_from_scratch(
    y_test,
    y_pred
)

f1 = f1_score_from_scratch(
    y_test,
    y_pred
)

tn, fp, fn, tp = confusion_matrix_from_scratch(
    y_test,
    y_pred
)


# ================================================================
# 10. DISPLAY MODEL PERFORMANCE
# ================================================================

print("\n")
print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(
    f"                 Predicted\n"
    f"                 Rejected  Approved\n"
    f"Actual Rejected     {tn:4d}      {fp:4d}\n"
    f"Actual Approved     {fn:4d}      {tp:4d}"
)


# ================================================================
# 11. LEARNED PARAMETERS
# ================================================================

print("\n")
print("=" * 70)
print("LEARNED PARAMETERS")
print("=" * 70)

for feature, weight in zip(
    X_df.columns,
    model.weights
):
    print(
        f"{feature:35s}: {weight: .6f}"
    )

print(
    f"{'Bias':35s}: {model.bias: .6f}"
)


# ================================================================
# 12. SAVE PREDICTIONS
# ================================================================

results = pd.DataFrame({
    "Actual Status": np.where(
        y_test == 1,
        "Approved",
        "Rejected"
    ),

    "Predicted Status": np.where(
        y_pred == 1,
        "Approved",
        "Rejected"
    ),

    "Approval Probability": y_prob,

    "Correct": (
        y_test == y_pred
    )
})

OUTPUT_FILE = "logistic_regression_output.csv"

results.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved:", OUTPUT_FILE)


# ================================================================
# 13. SHOW SAMPLE PREDICTIONS
# ================================================================

print("\n")
print("=" * 70)
print("SAMPLE PREDICTIONS")
print("=" * 70)

print(
    results.head(10).to_string(
        index=False
    )
)


# ================================================================
# 14. PLOT TRAINING LOSS
# ================================================================

plt.figure(figsize=(10, 5))

plt.plot(
    model.loss_history
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title(
    "Logistic Regression Training Loss"
)

plt.grid(True)

plt.show()


# ================================================================
# 15. CONFUSION MATRIX VISUALIZATION
# ================================================================

confusion_matrix = np.array([
    [tn, fp],
    [fn, tp]
])

plt.figure(figsize=(6, 5))

plt.imshow(
    confusion_matrix,
    interpolation="nearest"
)

plt.title("Confusion Matrix")

plt.colorbar()

plt.xticks(
    [0, 1],
    ["Rejected", "Approved"]
)

plt.yticks(
    [0, 1],
    ["Rejected", "Approved"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):

        plt.text(
            j,
            i,
            confusion_matrix[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()

plt.show()


# ================================================================
# 16. FINAL EXPERIMENT SUMMARY
# ================================================================

print("\n")
print("=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)

print("Algorithm       : Logistic Regression")
print("Training Method : Gradient Descent")
print("Implementation  : From Scratch using NumPy")
print(f"Input Features  : {X.shape[1]}")
print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples : {len(X_test)}")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1 Score        : {f1:.4f}")
print("=" * 70)